# A1 — Cadrer et explorer

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| A — Recommandation de films | 1 sur 4 | 3 h | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

La médiathèque de la ville lance un service de films en ligne pour ses abonnés. Chaque abonné peut noter les films qu'il a vus, de 0,5 à 5 étoiles. La responsable du service voudrait afficher, pour chacun, une sélection de films susceptibles de lui plaire, y compris pour les nouveaux abonnés qui n'ont encore noté que quelques films.

**La question ouverte : quels films proposer à un spectateur qu'on connaît peu ?**

Cette question est trop vaste pour être attaquée d'un bloc. Le travail de ce projet commence par la **découper** en sous-problèmes, dont chacun peut être résolu, vérifié et amélioré séparément.

![Schéma : les quatre étapes du projet, de A1 cadrer et explorer à A4 évaluer et livrer, avec les modules réinvestis](../../../assets/schemas/schema-projet-a-etapes.svg)

**Activité** : Découpez la question en trois ou quatre sous-questions précises, auxquelles un calcul peut répondre. Par exemple : « prévoir la note qu'un spectateur donnerait à un film qu'il n'a pas vu ». Pour chacune, indiquez comment on saurait que la réponse est bonne.

Indiquez vos réponses et justifiez-les.

**Activité** : Quelles informations seraient utiles, mais **manquent** dans un simple tableau de notes ? (Pensez au spectateur, au film, au contexte de visionnage, aux films vus mais non notés.) Lesquelles pourrait-on raisonnablement obtenir ?

Indiquez vos réponses et justifiez-les.

### Les données

Le projet utilise le jeu **MovieLens** (version réduite : environ 100 000 notes de 600 spectateurs sur 9 000 films), publié par le laboratoire GroupLens de l'université du Minnesota pour la recherche et l'enseignement. Sa licence en interdit la redistribution : le module `films.py`, fourni avec le projet, le télécharge sur votre ordinateur depuis le site de GroupLens. Sans connexion, il charge un **jeu simulé** de même format et de même taille, dont les résultats sont du même ordre (mais pas identiques).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from films import charger_films, decouper, reqm

notes, films, source = charger_films()
print("Source :", source)
print(f"{len(notes)} notes, {notes['userId'].nunique()} spectateurs, {notes['movieId'].nunique()} films notés")
notes.head()

## ② Estimer avant de calculer

In [ ]:
pari_proportion_cases_remplies = 0.10     # part des couples (spectateur, film) qui ont une note (modifiez)
pari_erreur_moyenne_naive = 0.5           # erreur typique, en étoiles, si l'on prédit la note moyenne (modifiez)

## ③ Abstraire : remplir les cases vides d'une matrice

On range les notes dans une matrice $R$ : une ligne par spectateur, une colonne par film. La plupart des cases sont vides. **Prévoir une note, c'est estimer une case vide.** Recommander, c'est ensuite choisir, sur la ligne d'un spectateur, les cases vides dont l'estimation est la plus haute.

![Schéma : une matrice spectateurs-films presque vide ; pour estimer une case, on part de la moyenne générale, corrigée par la sévérité du spectateur et la popularité du film](../../../assets/schemas/schema-matrice-notes.svg)

**Mesurer la qualité.** On met de côté une partie des notes connues, on les prédit sans les regarder, puis on compare avec la racine de l'erreur quadratique moyenne (module M6). Trois paquets :

- **apprentissage** : pour construire les modèles ;
- **validation** : pour comparer les modèles et choisir leurs réglages, dans les étapes A1 à A3 ;
- **test** : mis sous scellés jusqu'à l'étape A4, pour une évaluation finale honnête (module M6, notebook 6.4).

**Un premier modèle : les biais.** Certains spectateurs notent sévèrement, certains films plaisent à presque tout le monde :

$$
\hat r_{uf} = \mu + b_u + b_f .
$$

On choisit les biais qui minimisent

$$
\sum_{\text{notes connues}} \left(r_{uf} - \mu - b_u - b_f\right)^2 + \lambda\left(\sum_u b_u^2 + \sum_f b_f^2\right).
$$

Le second terme, la **régularisation**, empêche un film noté une seule fois 5 étoiles d'obtenir un biais énorme : sans information, on reste près de la moyenne. C'est un problème de moindres carrés (module M6) avec des milliers d'inconnues. On le résout en **alternant** : à films fixés, chaque $b_u$ a une formule simple, et inversement.

$$
b_u = \frac{\sum_{f} (r_{uf} - \mu - b_f)}{n_u + \lambda},
\qquad
b_f = \frac{\sum_{u} (r_{uf} - \mu - b_u)}{n_f + \lambda},
$$

où $n_u$ est le nombre de films notés par le spectateur $u$, et $n_f$ le nombre de notes du film $f$.

**Activité** : Un film n'a reçu qu'une note : 5 étoiles. La moyenne générale vaut 3,5. Que vaut $b_f$ (en prenant $b_u = 0$) avec $\lambda = 0$, puis avec $\lambda = 10$ ? Laquelle de ces deux valeurs vous semble la plus raisonnable ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Explorer

In [ ]:
n_spect, n_films = notes["userId"].nunique(), notes["movieId"].nunique()
remplissage = len(notes) / (n_spect * n_films)
print(f"Cases remplies : {remplissage:.2%}")

par_spectateur = notes.groupby("userId").size()
par_film = notes.groupby("movieId").size()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
notes["rating"].value_counts().sort_index().plot.bar(ax=axes[0], color=LIOUAI["vert"])
axes[0].set_title("Répartition des notes"); axes[0].set_xlabel("note")
axes[1].loglog(np.sort(par_spectateur.to_numpy())[::-1], color=LIOUAI["navy"])
axes[1].set_title("Notes par spectateur (classés)"); axes[1].set_xlabel("rang du spectateur")
axes[2].loglog(np.sort(par_film.to_numpy())[::-1], color=LIOUAI["navy"])
axes[2].set_title("Notes par film (classés)"); axes[2].set_xlabel("rang du film")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Médiane : {par_spectateur.median():.0f} notes par spectateur, {par_film.median():.0f} par film ; "
      f"{(par_film == 1).mean():.0%} des films n'ont qu'une note")

**Activité** : Décrivez ces trois graphiques en une phrase chacun. Que signifie la forme « en longue traîne » des deux derniers pour notre projet ? Pourquoi les spectateurs donnent-ils plus souvent de bonnes notes que de mauvaises ?

Indiquez vos réponses et justifiez-les.

### Découper les données

In [ ]:
reste, test = decouper(notes, part_test=0.2, graine=0)       # le test est mis sous scellés jusqu'à A4
app, val = decouper(reste, part_test=0.2, graine=1)
print(f"apprentissage {len(app)}, validation {len(val)}, test {len(test)} (sous scellés)")

### Trois modèles de référence

In [ ]:
mu = app["rating"].mean()
resultats = {"moyenne générale": reqm(val["rating"], np.full(len(val), mu))}

moyenne_film = app.groupby("movieId")["rating"].mean()
resultats["moyenne du film"] = reqm(val["rating"], val["movieId"].map(moyenne_film).fillna(mu))

def ajuster_biais(notes, lam=10.0, iterations=15):
    mu = notes["rating"].mean()
    u, f, r = notes["userId"].to_numpy(), notes["movieId"].to_numpy(), notes["rating"].to_numpy()
    bf = pd.Series(0.0, index=np.unique(f))
    for _ in range(iterations):
        g = pd.Series(r - mu - bf.loc[f].to_numpy()).groupby(u)
        bu = g.sum() / (g.count() + lam)                          # formule de b_u, films fixés
        g = pd.Series(r - mu - bu.loc[u].to_numpy()).groupby(f)
        bf = g.sum() / (g.count() + lam)                          # formule de b_f, spectateurs fixés
    return mu, bu.to_dict(), bf.to_dict()

def predire_biais(modele, notes):
    mu, bu, bf = modele
    return np.array([mu + bu.get(u, 0.0) + bf.get(f, 0.0) for u, f in zip(notes["userId"], notes["movieId"])])

modele = ajuster_biais(app, lam=10)
resultats["biais (λ = 10)"] = reqm(val["rating"], predire_biais(modele, val))
for nom, e in resultats.items():
    print(f"{nom:20s} REQM validation : {e:.4f}")

Les fonctions `ajuster_biais` et `predire_biais` sont reprises telles quelles dans `films.py` : les étapes suivantes les importeront.

### Et les spectateurs qu'on connaît peu ?

In [ ]:
nb_notes = app.groupby("userId").size()
classe = pd.cut(val["userId"].map(nb_notes), [0, 30, 60, 150, 10_000], labels=["≤ 30", "31 à 60", "61 à 150", "> 150"])
erreur = (val["rating"] - predire_biais(modele, val)) ** 2
print("REQM du modèle des biais selon le nombre de notes du spectateur (apprentissage) :")
print(np.sqrt(erreur.groupby(classe, observed=True).mean()).round(4).to_string())

### Recommander avec les biais

In [ ]:
_, bu, bf = modele
titres = films.set_index("movieId")["title"]
meilleurs = pd.Series(bf).sort_values(ascending=False)
print("Les 10 films au plus grand biais :")
for f, b in meilleurs.head(10).items():
    print(f"  {b:+.2f}  {titres.get(f, f)}  ({par_film.get(f, 0)} notes)")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari : {pari_proportion_cases_remplies:.0%} de cases remplies ; mesuré : {remplissage:.2%}")
print(f"Votre pari : erreur {pari_erreur_moyenne_naive} étoile ; mesuré : {resultats['moyenne générale']:.2f}")

**Activité** :

1. Énoncez le résultat de l'étape : quel modèle de référence retenez-vous, avec quelle erreur, en étoiles ?
2. Le modèle des biais recommande-t-il des films différents à deux spectateurs différents ? Pourquoi est-ce une limite majeure pour notre question ?
3. Les spectateurs peu actifs sont-ils moins bien prédits ? L'écart est-il important ?
4. Rédigez la liste de vos hypothèses de travail : par exemple, « un film non noté est un film non vu », « les goûts ne changent pas pendant la période ». Lesquelles sont discutables ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Un découpage parmi d'autres.** On a mis de côté des notes au hasard. Pour un service réel, on prédit l'**avenir** à partir du **passé**. Comment découper les données selon la date (colonne `timestamp`) ? Le résultat serait-il plus ou moins bon ?
- **Le réglage $\lambda$** a été choisi au jugé. C'est une constante qu'il faut traiter comme un paramètre.
- **Généraliser.** Écrivez une fonction `choisir_lambda(app, val, lambdas)` qui renvoie, pour chaque valeur de $\lambda$, l'erreur de validation, et la meilleure valeur. Tracez la courbe. Pourquoi ne faut-il surtout pas utiliser le paquet de test pour ce choix ?

In [ ]:
def choisir_lambda(app, val, lambdas):
    # À compléter : renvoie (erreurs, meilleur_lambda)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def choisir_lambda(app, val, lambdas):
    erreurs = {lam: reqm(val["rating"], predire_biais(ajuster_biais(app, lam), val)) for lam in lambdas}
    return erreurs, min(erreurs, key=erreurs.get)

erreurs, meilleur = choisir_lambda(app, val, [0, 1, 2, 5, 10, 20, 40, 80])
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.semilogx([max(l, 0.5) for l in erreurs], list(erreurs.values()), "o-", color=LIOUAI["navy"])
ax.set_xlabel("λ (0 affiché à 0,5)"); ax.set_ylabel("REQM validation"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Meilleur λ :", meilleur)

# Découpage temporel : les 20 % de notes les plus récentes de chaque spectateur
rang = notes.groupby("userId")["timestamp"].rank(pct=True)
passe, futur = notes[rang <= 0.8], notes[rang > 0.8]
print(f"Découpage temporel : REQM {reqm(futur['rating'], predire_biais(ajuster_biais(passe, meilleur), futur)):.4f}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Découper un gros problème en petits problèmes": 1,
    "Identifier l'information manquante": 1,
    "Formuler des questions précises": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Estimer la solution avant de calculer": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "A1-cadrer-et-explorer", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")